## Chapter 6 Altair Data Visualization
Exercise 2

In [ ]:
import sys; sys.path.insert(0, '..')
from src.ch6_init import (
    RAW_DATA_DIR, IMAGES_DIR,
    pd, np, alt,
    read_excel_safe,
    init_chapter6, display_init_status,
    ChartExportManager,
)
status = init_chapter6(notebook_name="ch6-exercise-2")
display_init_status(status)
exporter = status["export_manager"]

In [ ]:
src_file = RAW_DATA_DIR / 'EPA_fuel_economy_summary.csv'
df = pd.read_csv(src_file)
df.head()

In [ ]:
alt.Chart(df).mark_circle(size=50).encode(
    x='displ',
    y='fuelCost08',
    tooltip=['make', 'model', 'year'],
).interactive()

In [ ]:
chart1 = alt.Chart(df).mark_tick().encode(
    y='fuel_type_summary',
    x='barrels08'
)
chart2 = alt.Chart(df).mark_bar().encode(
    alt.X('barrels08:Q', bin=True),
    alt.Y('count()')
)
chart1 | chart2

In [ ]:
chart2 & chart1

In [ ]:
alt.hconcat(chart1, chart2)

In [ ]:
alt.vconcat(chart1, chart2)

In [ ]:
alt.Chart(df).mark_circle(size=50).encode(
    x='displ',
    y='fuelCost08',
    color='class_summary:N',
    tooltip=['make', 'model', 'year'],
).facet(row='class_summary:N')

In [ ]:
alt.Chart(df).mark_circle(size=50).encode(
    x='displ',
    y='fuelCost08',
    color='class_summary:N',
    tooltip=['make', 'model', 'year'],
).facet('class_summary:N', columns=2)

In [ ]:
base_chart = alt.Chart(df).mark_circle(size=50).encode(
    x='displ',
    y='fuelCost08',
    color='class_summary:N',
    tooltip=['make', 'model', 'year'],
)

base_chart.facet('class_summary:N', columns=2)

In [ ]:
bars = alt.Chart(df).mark_bar().encode(
    x='mean(fuelCost08):Q',
    y='year:O'
)
bars

In [ ]:
rule = alt.Chart(df).mark_rule(color='red').encode(
    x='mean(fuelCost08):Q'
)
bars + rule

In [ ]:
text = bars.mark_text(
    align='left', dx=3).encode(text=alt.Text('mean(fuelCost08):Q', format=',.0f'))
(bars + rule + text)

In [ ]:
combined_bar_chart = (bars + rule + text).properties(width=700)
combined_bar_chart

---

### Exporting Charts with ChartExportManager

Use the ``ChartExportManager`` (initialized in the setup cell) to export
any chart from this notebook in a consistent way. The examples below
reuse chart variables created earlier in this notebook.

In [ ]:
result = exporter.export(
    combined_bar_chart,
    "bar_yearly_mean_fuelcost",
    formats=("html", "png", "svg"),
    display=True
)

print(f"Export success: {result['success']}")
if not result['success']:
    print(f"Failed formats: {result['errors']}")

In [ ]:
facet_chart = base_chart.facet('class_summary:N', columns=2)

batch_results = exporter.export_batch({
    "tick_fueltype_barrels": chart1,
    "bar_barrels_histogram": chart2,
    "facet_class_scatter": facet_chart,
})

for r in batch_results:
    status = 'OK' if r['success'] else 'FAILED'
    print(f"{r['chart_id']}: {status}")
    if not r['success']:
        print(f"  Errors: {r['errors']}")

In [ ]:
exporter.display_status()

In [ ]:
print("All exported files:")
for path in exporter.list_exports():
    print(f"  {path.name}")